# Python Practical Patterns

## Part 1: Property Setters for Data Validation

### 1. Using Setter Property to Enforce Maximum Value Constraint

**What are Property Setters?**
- Property setters allow you to control how values are assigned to attributes
- They enable data validation before assignment
- They provide a clean interface while maintaining encapsulation
- Useful for enforcing business rules and constraints

**Why Use Property Setters for Validation?**
- Prevent invalid data from being stored
- Maintain data integrity
- Provide meaningful error messages
- Hide validation logic from the user of the class

In [1]:
class ValueContainer:
    def __init__(self, max_value=100):
        self._value = 0
        self._max_value = max_value
    
    @property
    def value(self):
        return self._value
    
    @value.setter
    def value(self, new_value):
        if new_value > self._max_value:
            raise ValueError(f"Value cannot exceed {self._max_value}")
        if new_value < 0:
            raise ValueError("Value cannot be negative")
        self._value = new_value
    
    @property
    def max_value(self):
        return self._max_value
    
    @max_value.setter
    def max_value(self, new_max):
        if new_max < 0:
            raise ValueError("Max value cannot be negative")
        self._max_value = new_max
        # Adjust current value if it exceeds new max
        if self._value > self._max_value:
            self._value = self._max_value

# Usage example
container = ValueContainer(max_value=100)
print(f"Initial value: {container.value}")
print(f"Max value: {container.max_value}")

# Valid assignment
container.value = 50
print(f"After setting to 50: {container.value}")

# Try to exceed max value
try:
    container.value = 150
except ValueError as e:
    print(f"Error: {e}")

# Try negative value
try:
    container.value = -10
except ValueError as e:
    print(f"Error: {e}")

Initial value: 0
Max value: 100
After setting to 50: 50
Error: Value cannot exceed 100
Error: Value cannot be negative


### 2. Practical Example: Bank Account with Balance Constraint

This example shows a realistic use case for property setters in a financial application.

In [10]:
class BankAccount:
    def __init__(self, account_number, initial_balance=0, overdraft_limit=0):
        self.account_number = account_number
        self._balance = initial_balance
        self._overdraft_limit = overdraft_limit
    
    @property
    def balance(self):
        return self._balance
    
    @balance.setter
    def balance(self, new_balance):
        if new_balance < -self._overdraft_limit:
            raise ValueError(f"Balance cannot exceed overdraft limit of {self._overdraft_limit}")
        self._balance = new_balance
    
    @property
    def overdraft_limit(self):
        return self._overdraft_limit
    
    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit amount must be positive")
        self.balance += amount
        print(f"Deposited {amount}. New balance: {self.balance}")
    
    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive")
        if self.balance - amount < -self._overdraft_limit:
            raise ValueError("Insufficient funds")
        self.balance -= amount
        print(f"Withdrew {amount}. New balance: {self.balance}")

# Usage
account = BankAccount("12345", initial_balance=1000, overdraft_limit=500)
print(f"Initial balance: {account.balance}")

account.deposit(500)
account.withdraw(200)

# Try to withdraw more than allowed
try:
    account.withdraw(2000)
except ValueError as e:
    print(f"Error: {e}")

Initial balance: 1000
Deposited 500. New balance: 1500
Withdrew 200. New balance: 1300
Error: Insufficient funds


## Part 2: Environment Variables for Configuration Management

### 1. Introduction to Environment Variables

**What are Environment Variables?**
- Key-value pairs stored in the operating system environment
- Used to configure applications without modifying code
- Separate configuration from code (12-factor app methodology)
- Secure way to store sensitive information (API keys, passwords)

**Why Use Environment Variables?**
- **Security**: Keep sensitive data out of source code
- **Flexibility**: Change configuration without redeploying
- **Portability**: Different configs for dev, staging, production
- **Best Practice**: Industry standard for configuration management

In [11]:
import os

# Basic environment variable access
print("=== Basic Environment Variable Access ===")

# Get environment variable (returns None if not found)
home_dir = os.getenv('HOME')
print(f"HOME: {home_dir}")

# Get with default value
app_name = os.getenv('APP_NAME', 'MyApp')
print(f"APP_NAME: {app_name}")

# Get using os.environ (raises KeyError if not found)
try:
    path = os.environ['PATH']
    print(f"PATH : {path} characters")
    print(f"PATH length: {len(path)} characters")
except KeyError as e:
    print(f"Environment variable not found: {e}")

# Set environment variable (only for current process)
os.environ['MY_VAR'] = 'Hello from Python'
print(f"MY_VAR: {os.getenv('MY_VAR')}")

=== Basic Environment Variable Access ===
HOME: None
APP_NAME: MyApp
PATH : c:\Python314;c:\Users\Kalyan Bandaru\AppData\Roaming\Python\Python314\Scripts;C:\Users\Kalyan Bandaru\AppData\Local\Programs\Windsurf;C:\Python314\Scripts\;C:\Python314\;C:\WINDOWS\system32;C:\WINDOWS;C:\WINDOWS\System32\Wbem;C:\WINDOWS\System32\WindowsPowerShell\v1.0\;C:\WINDOWS\System32\OpenSSH\;C:\Program Files\dotnet\;c:\Users\Kalyan Bandaru\AppData\Local\Programs\cursor\resources\app\bin;C:\Program Files\Docker\Docker\resources\bin;C:\Program Files\nodejs\;C:\ProgramData\chocolatey\bin;C:\Program Files\Git\cmd;C:\Users\Kalyan Bandaru\AppData\Local\Programs\Python\Launcher\;C:\Users\Kalyan Bandaru\AppData\Local\Microsoft\WindowsApps;C:\Users\Kalyan Bandaru\AppData\Local\Programs\Microsoft VS Code\bin;C:\Users\Kalyan Bandaru\AppData\Local\Programs\Windsurf\bin;C:\Users\Kalyan Bandaru\AppData\Roaming\npm;C:\Users\Kalyan Bandaru\AppData\Local\Programs\Windsurf;C:\Python314\Scripts\;C:\Python314\;C:\WINDOWS\sys

### 2. Configuration Management with Environment Variables

**Best Practices:**
- Use a configuration class to manage all environment variables
- Provide sensible defaults for non-critical values
- Validate required variables at startup
- Type-convert string values to appropriate types
- Use .env files for local development (never commit them)

In [12]:
class Config:
    """Configuration class that loads settings from environment variables."""
    
    # Database configuration
    DATABASE_URL = os.getenv('DATABASE_URL', 'sqlite:///default.db')
    DATABASE_POOL_SIZE = int(os.getenv('DATABASE_POOL_SIZE', '5'))
    
    # API configuration
    API_KEY = os.getenv('API_KEY', '')
    API_TIMEOUT = int(os.getenv('API_TIMEOUT', '30'))
    API_MAX_RETRIES = int(os.getenv('API_MAX_RETRIES', '3'))
    
    # Application configuration
    DEBUG = os.getenv('DEBUG', 'False').lower() == 'true'
    LOG_LEVEL = os.getenv('LOG_LEVEL', 'INFO').upper()
    PORT = int(os.getenv('PORT', '8000'))
    HOST = os.getenv('HOST', '0.0.0.0')
    
    # Feature flags
    ENABLE_FEATURE_X = os.getenv('ENABLE_FEATURE_X', 'False').lower() == 'true'
    ENABLE_FEATURE_Y = os.getenv('ENABLE_FEATURE_Y', 'True').lower() == 'true'
    
    @classmethod
    def validate(cls):
        """Validate that required configuration is present."""
        required_vars = {
            'API_KEY': cls.API_KEY,
        }
        
        missing = [var for var, value in required_vars.items() if not value]
        if missing:
            raise ValueError(f"Required environment variables missing: {missing}")
        
        # Validate types
        if cls.PORT < 1 or cls.PORT > 65535:
            raise ValueError("PORT must be between 1 and 65535")
        
        if cls.API_TIMEOUT < 1:
            raise ValueError("API_TIMEOUT must be positive")
        
        print("Configuration validated successfully!")
        return True
    
    @classmethod
    def display(cls):
        """Display current configuration (hide sensitive values)."""
        print("=== Configuration ===")
        print(f"DATABASE_URL: {cls.DATABASE_URL}")
        print(f"API_KEY: {'*' * 20 if cls.API_KEY else 'NOT SET'}")
        print(f"API_TIMEOUT: {cls.API_TIMEOUT}")
        print(f"DEBUG: {cls.DEBUG}")
        print(f"PORT: {cls.PORT}")
        print(f"HOST: {cls.HOST}")
        print(f"ENABLE_FEATURE_X: {cls.ENABLE_FEATURE_X}")
        print(f"ENABLE_FEATURE_Y: {cls.ENABLE_FEATURE_Y}")

# Example usage
Config.display()

# Try to validate (will fail if API_KEY is not set)
try:
    Config.validate()
except ValueError as e:
    print(f"Validation error: {e}")

# Set API_KEY and validate again
os.environ['API_KEY'] = 'test_api_key_12345'
Config.API_KEY = os.getenv('API_KEY')
Config.validate()

=== Configuration ===
DATABASE_URL: postgresql://prod-db.example.com/mydb
API_KEY: ********************
API_TIMEOUT: 30
DEBUG: False
PORT: 8000
HOST: 0.0.0.0
ENABLE_FEATURE_X: False
ENABLE_FEATURE_Y: True
Configuration validated successfully!
Configuration validated successfully!


True

### 3. Using .env Files for Local Development

**Note:** For production use the `python-dotenv` package to load .env files.

**Install:** `pip install python-dotenv`

**Example .env file:**
```
DATABASE_URL=postgresql://localhost/mydb
API_KEY=your_api_key_here
API_TIMEOUT=30
DEBUG=True
PORT=5000
```

**Important:** Never commit .env files to version control! Add `.env` to your `.gitignore`.

In [8]:
# Simulating python-dotenv behavior
def load_env_file(env_path):
    """Load environment variables from a .env file."""
    try:
        with open(env_path, 'r') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#') and '=' in line:
                    key, value = line.split('=', 1)
                    os.environ[key.strip()] = value.strip()
        print(f"Loaded environment variables from {env_path}")
    except FileNotFoundError:
        print(f"No .env file found at {env_path}")

# In real projects, use:
# from dotenv import load_dotenv
# load_dotenv()

# Example of creating a .env file programmatically
env_content = '''
# Database configuration
DATABASE_URL=postgresql://localhost/mydb
DATABASE_POOL_SIZE=10

# API configuration
API_KEY=my_secret_api_key
API_TIMEOUT=60
API_MAX_RETRIES=5

# Application configuration
DEBUG=True
LOG_LEVEL=DEBUG
PORT=5000
HOST=0.0.0.0

# Feature flags
ENABLE_FEATURE_X=True
ENABLE_FEATURE_Y=False
'''

# Write to .env file (for demonstration)
with open('.env.example', 'w') as f:
    f.write(env_content)

print("Created .env.example file with template configuration")
print("\nTo use in your project:")
print("1. Copy .env.example to .env")
print("2. Update values in .env with your actual configuration")
print("3. Add .env to .gitignore")
print("4. Load with: from dotenv import load_dotenv; load_dotenv()")

Created .env.example file with template configuration

To use in your project:
1. Copy .env.example to .env
2. Update values in .env with your actual configuration
3. Add .env to .gitignore
4. Load with: from dotenv import load_dotenv; load_dotenv()


### 4. Advanced Pattern: Environment-Specific Configuration

This pattern allows different configurations for development, testing, and production environments.

In [9]:
class EnvironmentConfig:
    """Environment-specific configuration."""
    
    ENV = os.getenv('ENV', 'development').lower()
    
    # Development defaults
    DEV_CONFIG = {
        'database_url': 'sqlite:///dev.db',
        'debug': True,
        'log_level': 'DEBUG',
    }
    
    # Testing defaults
    TEST_CONFIG = {
        'database_url': 'sqlite:///test.db',
        'debug': True,
        'log_level': 'DEBUG',
    }
    
    # Production defaults
    PROD_CONFIG = {
        'database_url': os.getenv('DATABASE_URL'),
        'debug': False,
        'log_level': 'INFO',
    }
    
    @classmethod
    def get_config(cls):
        """Get configuration based on current environment."""
        if cls.ENV == 'production':
            return cls.PROD_CONFIG
        elif cls.ENV == 'testing':
            return cls.TEST_CONFIG
        else:
            return cls.DEV_CONFIG
    
    @classmethod
    def setup(cls):
        """Setup configuration for current environment."""
        config = cls.get_config()
        
        print(f"=== Setting up for {cls.ENV.upper()} environment ===")
        print(f"Database URL: {config['database_url']}")
        print(f"Debug Mode: {config['debug']}")
        print(f"Log Level: {config['log_level']}")
        
        return config

# Test different environments
print("=== Development Environment ===")
os.environ['ENV'] = 'development'
EnvironmentConfig.ENV = os.getenv('ENV', 'development').lower()
dev_config = EnvironmentConfig.setup()

print("\n=== Testing Environment ===")
os.environ['ENV'] = 'testing'
EnvironmentConfig.ENV = os.getenv('ENV', 'development').lower()
test_config = EnvironmentConfig.setup()

print("\n=== Production Environment ===")
os.environ['ENV'] = 'production'
os.environ['DATABASE_URL'] = 'postgresql://prod-db.example.com/mydb'
EnvironmentConfig.ENV = os.getenv('ENV', 'development').lower()
prod_config = EnvironmentConfig.setup()

=== Development Environment ===
=== Setting up for DEVELOPMENT environment ===
Database URL: sqlite:///dev.db
Debug Mode: True
Log Level: DEBUG

=== Testing Environment ===
=== Setting up for TESTING environment ===
Database URL: sqlite:///test.db
Debug Mode: True
Log Level: DEBUG

=== Production Environment ===
=== Setting up for PRODUCTION environment ===
Database URL: None
Debug Mode: False
Log Level: INFO


## Summary

### Property Setters for Validation:
- Use `@property` decorator to create getters
- Use `@attribute.setter` decorator to create setters
- Enforce constraints (min/max values, type checking, business rules)
- Provide meaningful error messages for invalid input

### Environment Variables for Configuration:
- Use `os.getenv()` to read environment variables
- Provide sensible defaults for non-critical values
- Validate required variables at application startup
- Use .env files for local development (never commit them)
- Separate configuration for different environments
- Keep sensitive data (API keys, passwords) out of source code

# Chapter 1 Q&A Practice

These questions are imported from `1250 Python Q&A - Copy.xlsx` for Chapter 1. Use them after studying the concept sections above.


## Environment Variables

### Q75. How to use environment variables with os for config management?

- **Difficulty:** Medium
- **Estimated effort:** 10–15 minutes
- **Why it matters:** Secure config handling

**Answer outline:** Explain name binding, dynamic typing, naming rules, PEP 8 style, and the convention of uppercase names for constants.

